In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
from openai import OpenAI
from google.colab import userdata

In [ ]:
# FIX (Bug 2): Use the Instruct model, not the base model.
# The base model ("Meta-Llama-3.1-8B") won't reliably follow instructions
# or produce structured output. The Instruct variant was fine-tuned for that.
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None          # Auto-detect: float16 for T4/V100, bfloat16 for Ampere+
load_in_4bit = True   # 4-bit quantization to fit in GPU memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",  # <-- Instruct model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
==((====))==  Unsloth 2026.3.5: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
# FIX (Bug 2 continued): LoRA / PEFT setup REMOVED.
# get_peft_model() adds trainable adapter weights — that's for fine-tuning,
# not inference. We're using the Instruct model as-is, so we skip this entirely.

# Instead, put the model into fast inference mode:
FastLanguageModel.for_inference(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm):

In [ ]:
# The alpaca_prompt template and formatting_prompts_func from the Unsloth
# fine-tuning tutorial are not needed for inference-only judging.
# (Removed to avoid confusion — we build our evaluation prompt directly in cell 7.)

In [ ]:
df=pd.read_csv("results_gpt_econ.csv")
print(f"Total rows in CSV: {len(df)}")

In [ ]:
df.head()

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

def fetch_html_body_content(html_url):

    try:
        r = requests.get(html_url, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print("⚠ HTML 请求失败:", e)
        return None, "html_fetch_failed"

    body = soup.find("body")
    if not body:
        return None, "no_body"

    lines = [line.strip() for line in body.stripped_strings if line.strip()]
    text = "\n".join(lines)

    if not text:
        return None, "empty_text"

    # 移除 Abstract
    text = re.sub(
        r'Abstract[:\s].*?(?=(Introduction|1\.\sIntroduction))',
        '',
        text,
        flags=re.S|re.I
    )

    # 截断 Acknowledgment
    ack_patterns = [
        r'Acknowledgment',
        r'Acknowledgements',
        r'ACKNOWLEDGMENT',
        r'ACKNOWLEDGEMENTS'
    ]

    for pat in ack_patterns:
        match = re.search(pat, text, re.I)
        if match:
            text = text[:match.start()]
            break

    text = "\n".join(line for line in text.splitlines() if line.strip())

    return text, "html_success"

In [ ]:
import pandas as pd
import re
# from pipeline import fetch_html_body_content

df["llama_rate"] = pd.NA
df["llama_explanation"] = ""

# --- DATA QUALITY FIX: strip arXiv HTML navigation boilerplate ---
# arXiv HTML pages start with a big block of UI elements and a table-of-contents
# (section number + title lines) before the actual article prose. We need to skip
# past all of that to give Llama clean article text.

# Exact strings that appear in the arXiv HTML nav/UI preamble
BOILERPLATE_EXACT = {
    "Report GitHub Issue", "Submit without GitHub", "Submit in GitHub",
    "Back to arXiv", "Why HTML?", "Report Issue", "Content selection saved",
    "Describe the issue below", "×",
}

# Substrings — if ANY of these appear anywhere in the line, skip it
BOILERPLATE_SUBSTRINGS = [
    "Report GitHub Issue",
    "Back to arXiv",
    "Why HTML?",
    "Report Issue",
    "Submit without GitHub",
    "Submit in GitHub",
    "Back to Introduction",
    "Back to ",            # catches "Back to <any section>"
    "Content selection saved",
    "Describe the issue below",
]

def get_article_snippet(full_text, max_chars=4000):
    """
    Strip the arXiv HTML preamble (nav UI + table of contents), then return
    the first max_chars of actual article prose.

    Strategy: scan forward until we find a line that looks like the start of
    real article content (a sentence-length line with lowercase words — i.e.
    actual prose, not a short section heading). Everything before that is
    preamble/TOC and gets dropped.
    """
    lines = full_text.splitlines()

    # Phase 1: Find where the real article prose begins.
    # TOC lines are short (section titles). Real prose lines are longer and
    # contain lowercase words forming sentences.
    body_start = 0
    for i, line in enumerate(lines):
        stripped = line.strip()
        # Skip blanks
        if not stripped:
            continue
        # A "prose" line: reasonably long, contains spaces (multiple words),
        # and has some lowercase characters (not all-caps headings).
        if len(stripped) > 80 and " " in stripped and re.search(r"[a-z]", stripped):
            body_start = i
            break

    # Phase 2: From body_start onward, drop any remaining boilerplate lines
    cleaned_lines = []
    for line in lines[body_start:]:
        stripped = line.strip()
        if not stripped:
            # Preserve paragraph breaks in the body, but skip leading blanks
            if cleaned_lines:
                cleaned_lines.append("")
            continue
        # Skip lines matching known boilerplate substrings
        if any(bp in stripped for bp in BOILERPLATE_SUBSTRINGS):
            continue
        # Skip exact boilerplate matches
        if stripped in BOILERPLATE_EXACT:
            continue
        # Skip bare section numbers like "2", "2.1", "A.3"
        if re.match(r"^[A-Z]?\.?\d+(\.\d+)*$", stripped):
            continue
        cleaned_lines.append(line)

    cleaned = "\n".join(cleaned_lines)
    return cleaned[:max_chars]


# --- Main evaluation loop ---
for index, row in df.iterrows():
    try:
        html_url = row.get("html_url", None)
        if not html_url:
            continue
        article, status = fetch_html_body_content(html_url)
        if status != "html_success":
            print(f"Row {index}: HTML fetch failed ({status})")
            continue

        article_snippet = get_article_snippet(article)
        summary = row["gpt_prompt"]

        # FIX: Use Llama 3.1 Instruct's native chat template.
        # The Instruct model was fine-tuned on this specific format. Feeding a
        # raw string causes the model to treat it as text-to-continue rather
        # than an instruction to follow. tokenizer.apply_chat_template() builds
        # the correct <|begin_of_text|><|start_header_id|>... structure.
        messages = [
            {"role": "system", "content": (
                "You are an expert at evaluating scientific news summaries. "
                "You will be given an article excerpt and a summary. "
                "Evaluate the summary and respond with EXACTLY two lines:\n"
                "Explanation: <your rationale in complete sentences>\n"
                "Total rating: <integer 1-5>\n"
                "Do not output anything else."
            )},
            {"role": "user", "content": (
                f"Article:\n{article_snippet}\n\n"
                f"Summary:\n{summary}"
            )},
        ]

        encoded = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,   # adds the assistant turn header
            truncation=True,
            max_length=max_seq_length,
        ).to("cuda")

        # FIX (Bug 1): max_new_tokens raised from 64 → 400
        outputs = model.generate(
            input_ids=encoded,
            max_new_tokens=400,
            use_cache=True,
        )
        output_text = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )[0].strip()

        print(f"--- Row {index} ---")
        print(output_text)
        print()

        # --- Parse Explanation and Total rating from model output ---
        explanation, rating = "", pd.NA
        for line in output_text.splitlines():
            line = line.strip()
            if line.lower().startswith("explanation:"):
                explanation = line.split(":", 1)[1].strip()
            elif line.lower().startswith("total rating:"):
                match = re.search(r"\d+", line)
                if match:
                    rating = int(match.group())

        df.at[index, "llama_rate"] = rating
        df.at[index, "llama_explanation"] = explanation
        print(f"Row {index} done — rating={rating}")
    except Exception as e:
        print(f"Error at row {index}: {e}")

# Save results
df.to_csv("results_llama_econ_with_ratings.csv", index=False)
print(f"\nSaved {len(df)} rows to results_llama_phy_with_ratings.csv")